# Import

In [ ]:
import os
import random
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

# CONFIGURATION

In [ ]:
YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

TERMS = {
    "SP": {"name": "Spring", "months": [1, 2, 3, 4, 5], "min_files": 40, "max_files": 60},
    "SU": {"name": "Summer", "months": [3, 4, 5, 6, 7, 8], "min_files": 70, "max_files": 90},
    "FA": {"name": "Fall",   "months": [4, 5, 6, 7, 8, 9, 10, 11], "min_files": 120, "max_files": 160}
}


# Processing

In [ ]:

# EXACT MATCH: Enrollment Statuses based on your Main Summary file
ENROLLMENT_STATUSES = [
    "Continuing", 
    "Continuing CAP", 
    "Continuing Non-Matric", 
    "Continuing PTech", 
    "New CAP", 
    "New First-Time", 
    "New Non-Matric", 
    "New Transfer", 
    "Cross Registered"
]

def create_fte_excel(filepath, term_display, run_date):
    """Creates a formatted Excel file matching the required FTE template."""
    wb = Workbook()
    ws = wb.active
    ws.title = "FTE"

    # --- ROW 1: Metadata ---
    # Writes exact shortcode format like "Term: FA-20"
    ws['A1'] = f"Term: {term_display}"
    ws['A1'].font = Font(bold=True)
    ws['F1'] = f"Run Date: {run_date.strftime('%m/%d/%Y')}"
    ws['F1'].font = Font(bold=True)

    # --- ROW 3: Main Headers ---
    ws['A3'] = "Enrollment Status"
    ws['B3'] = "Full Time"
    ws.merge_cells('B3:D3')
    ws['E3'] = "Part Time"
    ws.merge_cells('E3:G3')
    ws['H3'] = "Grand Total"
    ws.merge_cells('H3:J3')

    for cell in ['A3', 'B3', 'E3', 'H3']:
        ws[cell].font = Font(bold=True)
        ws[cell].alignment = Alignment(horizontal="center")

    # --- ROW 4: Sub Headers ---
    sub_headers = ["", "#", "Credits", "FTE", "#", "Credits", "FTE", "#", "Credits", "FTE"]
    for col_idx, val in enumerate(sub_headers, start=1):
        cell = ws.cell(row=4, column=col_idx, value=val)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")

    # --- ROW 5+: Data Generation ---
    row_idx = 5
    sums = {"FT_#": 0, "FT_CR": 0, "FT_FTE": 0, "PT_#": 0, "PT_CR": 0, "PT_FTE": 0}

    for status in ENROLLMENT_STATUSES:
        ft_num = random.randint(0, 150)
        ft_cred = ft_num * random.randint(12, 16)
        ft_fte = round(ft_cred / 15, 2)

        pt_num = random.randint(0, 150)
        pt_cred = pt_num * random.randint(3, 11)
        pt_fte = round(pt_cred / 15, 2)

        gt_num = ft_num + pt_num
        gt_cred = ft_cred + pt_cred
        gt_fte = ft_fte + pt_fte

        sums["FT_#"] += ft_num
        sums["FT_CR"] += ft_cred
        sums["FT_FTE"] += ft_fte
        sums["PT_#"] += pt_num
        sums["PT_CR"] += pt_cred
        sums["PT_FTE"] += pt_fte

        ws.cell(row=row_idx, column=1, value=status)
        ws.cell(row=row_idx, column=2, value=ft_num)
        ws.cell(row=row_idx, column=3, value=ft_cred)
        ws.cell(row=row_idx, column=4, value=ft_fte)
        ws.cell(row=row_idx, column=5, value=pt_num)
        ws.cell(row=row_idx, column=6, value=pt_cred)
        ws.cell(row=row_idx, column=7, value=pt_fte)
        ws.cell(row=row_idx, column=8, value=gt_num)
        ws.cell(row=row_idx, column=9, value=gt_cred)
        ws.cell(row=row_idx, column=10, value=gt_fte)
        
        row_idx += 1

    # --- Final Row: Grand Total ---
    ws.cell(row=row_idx, column=1, value="Grand Total").font = Font(bold=True)
    ws.cell(row=row_idx, column=2, value=sums["FT_#"]).font = Font(bold=True)
    ws.cell(row=row_idx, column=3, value=sums["FT_CR"]).font = Font(bold=True)
    ws.cell(row=row_idx, column=4, value=round(sums["FT_FTE"], 2)).font = Font(bold=True)
    ws.cell(row=row_idx, column=5, value=sums["PT_#"]).font = Font(bold=True)
    ws.cell(row=row_idx, column=6, value=sums["PT_CR"]).font = Font(bold=True)
    ws.cell(row=row_idx, column=7, value=round(sums["PT_FTE"], 2)).font = Font(bold=True)
    ws.cell(row=row_idx, column=8, value=sums["FT_#"] + sums["PT_#"]).font = Font(bold=True)
    ws.cell(row=row_idx, column=9, value=sums["FT_CR"] + sums["PT_CR"]).font = Font(bold=True)
    ws.cell(row=row_idx, column=10, value=round(sums["FT_FTE"] + sums["PT_FTE"], 2)).font = Font(bold=True)

    ws.column_dimensions['A'].width = 25
    wb.save(filepath)


def generate_structure_and_files(root_dir):
    """Builds folders and generates files in the specified directory."""
    if not os.path.exists(root_dir):
        os.makedirs(root_dir)

    total_file_count = 0

    for year in YEARS:
        year_str = str(year)
        year_path = os.path.join(root_dir, year_str)
        if not os.path.exists(year_path):
            os.makedirs(year_path)

        for term_code, term_info in TERMS.items():
            folder_term_name = f"{term_code}-{year_str[-2:]}"
            term_path = os.path.join(year_path, folder_term_name)
            if not os.path.exists(term_path):
                os.makedirs(term_path)

            # EXACT MATCH: Forces the internal file string to "FA-24" instead of "Fall 2024"
            term_display = f"{term_code}-{year_str[-2:]}"
            
            # Create a pool of all valid days for this term's active months
            possible_dates = []
            for m in term_info["months"]:
                for d in range(1, 29):  # 1st to 28th to avoid end-of-month bugs
                    possible_dates.append(datetime(year, m, d).date())
            
            # Decide exactly how many files to generate based on your ranges
            target_count = random.randint(term_info["min_files"], term_info["max_files"])
            target_count = min(target_count, len(possible_dates))
            
            # Randomly select UNIQUE dates
            selected_dates = random.sample(possible_dates, target_count)
            
            # Generate the files
            for run_date in selected_dates:
                filename = f"{year}.{run_date.month}.{run_date.day} {term_code}.xlsx"
                filepath = os.path.join(term_path, filename)
                
                create_fte_excel(filepath, term_display, run_date)
                total_file_count += 1
            
            print(f"Generated {target_count} files for {term_display}")
                
    print(f"\n✅ Successfully generated {total_file_count} total dummy Excel files inside '{root_dir}'!")


if __name__ == "__main__":
    
    target_directory = r""  # <--- Update this to your exact folder path
    
    print("--- Dummy Data Generator ---")
    print(f"Target Directory: {target_directory}\n")
    generate_structure_and_files(target_directory)